# make_pre_ind_table equivalence and speed test

This notebook keeps the legacy reference implementation and adds a faster implementation
that uses `numba.njit` (with a NumPy fallback when Numba is unavailable).

What it does:
- verifies semantic equivalence vs the legacy TensorFlow version
- benchmarks runtime of old vs NumPy vs Numba-NJIT implementations

In [1]:
import numpy as np
import tensorflow as tf
from time import perf_counter

try:
    from numba import njit
    HAS_NUMBA = True
except Exception:
    HAS_NUMBA = False
    def njit(*args, **kwargs):
        def _wrap(fn):
            return fn
        return _wrap

print('TensorFlow:', tf.__version__)
print('Numba available:', HAS_NUMBA)
np.random.seed(7)

2026-02-12 10:58:54.869548: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 10:58:54.907146: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-12 10:58:54.907187: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-12 10:58:54.908376: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-12 10:58:54.914932: I tensorflow/core/platform/cpu_feature_guar

TensorFlow: 2.15.0
Numba available: True


In [10]:
def make_pre_ind_table_old(indices, n_source_neurons=197613):
    """Legacy reference implementation (forces CPU to avoid GPU DenseBincount determinism errors)."""
    with tf.device('/GPU:0'):
        indices_tf = tf.convert_to_tensor(indices)
        pre_ids = indices_tf[:, 1]
        sort_idx = tf.argsort(pre_ids, axis=0)
        sorted_pre = tf.gather(pre_ids, sort_idx)
        counts = tf.math.bincount(tf.cast(sorted_pre, tf.int32), minlength=n_source_neurons)
        row_splits = tf.concat([[0], tf.cumsum(counts)], axis=0)
        return tf.RaggedTensor.from_row_splits(sort_idx, row_splits, validate=False)

In [15]:
def _validate_inputs(indices, n_source_neurons):
    n_source_neurons = int(n_source_neurons)
    if n_source_neurons <= 0:
        raise ValueError(f'`n_source_neurons` must be > 0, got {n_source_neurons}.')

    indices_np = np.asarray(indices)
    if indices_np.ndim != 2 or indices_np.shape[1] < 2:
        raise ValueError(f'`indices` must have shape [n_synapses, >=2], got {indices_np.shape}.')

    pre_ids = indices_np[:, 1].astype(np.int64, copy=False)
    invalid = (pre_ids < 0) | (pre_ids >= n_source_neurons)
    if np.any(invalid):
        bad = int(pre_ids[np.flatnonzero(invalid)[0]])
        raise ValueError(
            f'Presynaptic index {bad} is out of bounds for `n_source_neurons={n_source_neurons}`.'
        )
    return pre_ids, n_source_neurons


def make_pre_ind_table_numpy(indices, n_source_neurons=197613):
    """Deterministic NumPy implementation (stable sort + bincount + cumsum)."""
    pre_ids, n_source_neurons = _validate_inputs(indices, n_source_neurons)
    if pre_ids.size == 0:
        order_np = np.empty((0,), dtype=np.int32)
        row_splits_np = np.zeros((n_source_neurons + 1,), dtype=np.int64)
    else:
        order_np = np.argsort(pre_ids, kind='stable')
        counts_np = np.bincount(pre_ids[order_np], minlength=n_source_neurons)
        row_splits_np = np.empty((n_source_neurons + 1,), dtype=np.int64)
        row_splits_np[0] = 0
        np.cumsum(counts_np, dtype=np.int64, out=row_splits_np[1:])

    if order_np.size <= np.iinfo(np.int32).max:
        order_tf = tf.convert_to_tensor(order_np, dtype=tf.int32)
    else:
        order_tf = tf.convert_to_tensor(order_np, dtype=tf.int64)
    row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)
    return tf.RaggedTensor.from_row_splits(order_tf, row_splits_tf, validate=False)


@njit(cache=True)
def _build_csr_order_numba(pre_ids, n_source_neurons):
    """O(n_syn + n_source) stable bucket build for CSR order/row_splits."""
    n_syn = pre_ids.shape[0]
    counts = np.zeros(n_source_neurons, dtype=np.int64)
    for i in range(n_syn):
        counts[pre_ids[i]] += 1

    row_splits = np.empty(n_source_neurons + 1, dtype=np.int64)
    row_splits[0] = 0
    for i in range(n_source_neurons):
        row_splits[i + 1] = row_splits[i] + counts[i]

    write_ptr = np.empty(n_source_neurons, dtype=np.int64)
    for i in range(n_source_neurons):
        write_ptr[i] = row_splits[i]

    order = np.empty(n_syn, dtype=np.int64)
    for syn_idx in range(n_syn):
        p = pre_ids[syn_idx]
        pos = write_ptr[p]
        order[pos] = syn_idx
        write_ptr[p] = pos + 1

    return order, row_splits

def make_pre_ind_table_njit(indices, n_source_neurons=197613):
    """Fast implementation: numba.njit path if available; deterministic NumPy fallback otherwise."""
    pre_ids, n_source_neurons = _validate_inputs(indices, n_source_neurons)

    if pre_ids.size == 0:
        order_np = np.empty((0,), dtype=np.int32)
        row_splits_np = np.zeros((n_source_neurons + 1,), dtype=np.int64)
    elif HAS_NUMBA:
        order_np, row_splits_np = _build_csr_order_numba(pre_ids, n_source_neurons)
    else:
        # Safe deterministic fallback if numba is unavailable.
        order_np = np.argsort(pre_ids, kind='stable')
        counts_np = np.bincount(pre_ids[order_np], minlength=n_source_neurons)
        row_splits_np = np.empty((n_source_neurons + 1,), dtype=np.int64)
        row_splits_np[0] = 0
        np.cumsum(counts_np, dtype=np.int64, out=row_splits_np[1:])

    if order_np.size <= np.iinfo(np.int32).max:
        order_tf = tf.convert_to_tensor(order_np, dtype=tf.int32)
    else:
        order_tf = tf.convert_to_tensor(order_np, dtype=tf.int64)
    row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)
    return tf.RaggedTensor.from_row_splits(order_tf, row_splits_tf, validate=False)

In [16]:
def ragged_semantic_equal(rt_a, rt_b):
    if not np.array_equal(rt_a.row_splits.numpy(), rt_b.row_splits.numpy()):
        return False

    rows_a = rt_a.to_list()
    rows_b = rt_b.to_list()
    if len(rows_a) != len(rows_b):
        return False

    for ra, rb in zip(rows_a, rows_b):
        if sorted(ra) != sorted(rb):
            return False
    return True


def ragged_exact_equal(rt_a, rt_b):
    return (
        np.array_equal(rt_a.row_splits.numpy(), rt_b.row_splits.numpy()) and
        np.array_equal(rt_a.flat_values.numpy(), rt_b.flat_values.numpy())
    )

In [17]:
# Edge cases
edge_cases = [
    (np.zeros((0, 2), dtype=np.int64), 1),
    (np.array([[0, 0]], dtype=np.int64), 1),
    (np.array([[10, 0], [11, 0], [12, 0]], dtype=np.int64), 5),
    (np.array([[10, 2], [11, 1], [12, 2], [13, 4], [14, 1]], dtype=np.int64), 6),
]

for i, (indices, n_src) in enumerate(edge_cases):
    old_rt = make_pre_ind_table_old(indices, n_src)
    np_rt = make_pre_ind_table_numpy(indices, n_src)
    njit_rt = make_pre_ind_table_njit(indices, n_src)

    assert ragged_semantic_equal(old_rt, np_rt), f'NumPy edge case {i} failed semantic check'
    assert ragged_semantic_equal(old_rt, njit_rt), f'NJIT edge case {i} failed semantic check'

print(f'Passed {len(edge_cases)} edge cases for NumPy and NJIT implementations.')

Passed 4 edge cases for NumPy and NJIT implementations.


In [18]:
# Randomized equivalence
n_trials = 100
exact_np = 0
exact_njit = 0
rng = np.random.default_rng(1234)

for t in range(n_trials):
    n_src = int(rng.integers(1, 500))
    n_syn = int(rng.integers(0, 5000))

    pre = rng.integers(0, n_src, size=n_syn, dtype=np.int64)
    post = rng.integers(0, 100000, size=n_syn, dtype=np.int64)
    indices = np.stack([post, pre], axis=1)

    old_rt = make_pre_ind_table_old(indices, n_src)
    np_rt = make_pre_ind_table_numpy(indices, n_src)
    njit_rt = make_pre_ind_table_njit(indices, n_src)

    if not ragged_semantic_equal(old_rt, np_rt):
        raise AssertionError(f'NumPy random trial {t} failed semantic equivalence')
    if not ragged_semantic_equal(old_rt, njit_rt):
        raise AssertionError(f'NJIT random trial {t} failed semantic equivalence')

    if ragged_exact_equal(old_rt, np_rt):
        exact_np += 1
    if ragged_exact_equal(old_rt, njit_rt):
        exact_njit += 1

print(f'Random trials passed: {n_trials}/{n_trials}')
print(f'Exact flat_values order matches (NumPy): {exact_np}/{n_trials}')
print(f'Exact flat_values order matches (NJIT): {exact_njit}/{n_trials}')
print('Semantic equivalence is the required correctness property.')

Random trials passed: 100/100
Exact flat_values order matches (NumPy): 100/100
Exact flat_values order matches (NJIT): 100/100
Semantic equivalence is the required correctness property.


In [20]:
# Performance benchmark
def benchmark(fn, indices, n_src, n_repeat=5):
    times = []
    for _ in range(n_repeat):
        t0 = perf_counter()
        rt = fn(indices, n_src)
        _ = rt.row_splits.numpy()  # force materialization
        times.append(perf_counter() - t0)
    return np.array(times)

rng = np.random.default_rng(2026)
n_src = 200_000
n_syn = 100_000_000
pre = rng.integers(0, n_src, size=n_syn, dtype=np.int64)
post = rng.integers(0, 200_000, size=n_syn, dtype=np.int64)
indices = np.stack([post, pre], axis=1)

# Warm up numba compilation once (excluded from timing).
# if HAS_NUMBA:
#     _ = make_pre_ind_table_njit(indices[:10_000], n_src).row_splits.numpy()

# t_old = benchmark(make_pre_ind_table_old, indices, n_src, n_repeat=10)
# t_np = benchmark(make_pre_ind_table_numpy, indices, n_src, n_repeat=1)
# t_njit = benchmark(make_pre_ind_table_njit, indices, n_src, n_repeat=10)

print('Old (TF bincount on CPU):', t_old, 'mean=', t_old.mean())
print('NumPy (stable sort):   ', t_np, 'mean=', t_np.mean())
print('NJIT (count+scatter):  ', t_njit, 'mean=', t_njit.mean())

print('Speedup NumPy vs old :', t_old.mean() / t_np.mean())
print('Speedup NJIT  vs old :', t_old.mean() / t_njit.mean())
print('Speedup NJIT  vs NumPy:', t_np.mean() / t_njit.mean())

Old (TF bincount on CPU): [0.80724411 0.55588273 0.55079364 0.55177906 0.55504524 0.55466561
 0.55316054 0.55402627 0.55507931 0.55472646] mean= 0.5792402976999256
NumPy (stable sort):    [18.56412864] mean= 18.56412864199956
NJIT (count+scatter):   [1.73439253 1.7167712  1.75565304 1.76395348 1.69803328 1.70009399
 1.69707691 1.69801446 1.76086576 1.7706699 ] mean= 1.7295524553000177
Speedup NumPy vs old : 0.031202126901310627
Speedup NJIT  vs old : 0.33490762071130553
Speedup NJIT  vs NumPy: 10.73348691166428


In [ ]:
# Active-synapse access benchmark: int32 vs int64 flat_values
def make_pre_ind_table_with_dtype(indices, n_source_neurons=197613, order_dtype=np.int32):
    pre_ids, n_source_neurons = _validate_inputs(indices, n_source_neurons)

    if pre_ids.size == 0:
        order_np = np.empty((0,), dtype=order_dtype)
        row_splits_np = np.zeros((n_source_neurons + 1,), dtype=np.int64)
    else:
        order_np = np.argsort(pre_ids, kind='stable')
        counts_np = np.bincount(pre_ids, minlength=n_source_neurons)
        row_splits_np = np.empty((n_source_neurons + 1,), dtype=np.int64)
        row_splits_np[0] = 0
        np.cumsum(counts_np, dtype=np.int64, out=row_splits_np[1:])
        order_np = order_np.astype(order_dtype, copy=False)

    order_tf = tf.convert_to_tensor(order_np, dtype=tf.int32)
    row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int32)

    # row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)
    return tf.RaggedTensor.from_row_splits(order_tf, row_splits_tf, validate=False)

def make_pre_ind_table_with_dtype0(indices, n_source_neurons=197613, order_dtype=np.int32):
    pre_ids, n_source_neurons = _validate_inputs(indices, n_source_neurons)

    if pre_ids.size == 0:
        order_np = np.empty((0,), dtype=order_dtype)
        row_splits_np = np.zeros((n_source_neurons + 1,), dtype=np.int64)
    else:
        order_np = np.argsort(pre_ids, kind='stable')
        counts_np = np.bincount(pre_ids, minlength=n_source_neurons)
        row_splits_np = np.empty((n_source_neurons + 1,), dtype=np.int64)
        row_splits_np[0] = 0
        np.cumsum(counts_np, dtype=np.int64, out=row_splits_np[1:])
        order_np = order_np.astype(order_dtype, copy=False)

    order_tf = tf.convert_to_tensor(order_np, dtype=tf.int64)
    row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int32)

    # row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)
    return tf.RaggedTensor.from_row_splits(order_tf, row_splits_tf, validate=False)

def make_pre_ind_table_with_dtype2(indices, n_source_neurons=197613, order_dtype=np.int32):
    pre_ids, n_source_neurons = _validate_inputs(indices, n_source_neurons)

    if pre_ids.size == 0:
        order_np = np.empty((0,), dtype=order_dtype)
        row_splits_np = np.zeros((n_source_neurons + 1,), dtype=np.int64)
    else:
        order_np = np.argsort(pre_ids, kind='stable')
        counts_np = np.bincount(pre_ids, minlength=n_source_neurons)
        row_splits_np = np.empty((n_source_neurons + 1,), dtype=np.int64)
        row_splits_np[0] = 0
        np.cumsum(counts_np, dtype=np.int64, out=row_splits_np[1:])
        order_np = order_np.astype(order_dtype, copy=False)

    order_tf = tf.convert_to_tensor(order_np, dtype=tf.int32)

    row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)

    # row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)
    return tf.RaggedTensor.from_row_splits(order_tf, row_splits_tf, validate=False)

def make_pre_ind_table_with_dtype3(indices, n_source_neurons=197613, order_dtype=np.int32):
    pre_ids, n_source_neurons = _validate_inputs(indices, n_source_neurons)

    if pre_ids.size == 0:
        order_np = np.empty((0,), dtype=order_dtype)
        row_splits_np = np.zeros((n_source_neurons + 1,), dtype=np.int64)
    else:
        order_np = np.argsort(pre_ids, kind='stable')
        counts_np = np.bincount(pre_ids, minlength=n_source_neurons)
        row_splits_np = np.empty((n_source_neurons + 1,), dtype=np.int64)
        row_splits_np[0] = 0
        np.cumsum(counts_np, dtype=np.int64, out=row_splits_np[1:])
        order_np = order_np.astype(order_dtype, copy=False)

    order_tf = tf.convert_to_tensor(order_np, dtype=tf.int64)

    row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)

    # row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int64)
    return tf.RaggedTensor.from_row_splits(order_tf, row_splits_tf, validate=False)


def benchmark_build(indices, n_src, order_dtype, func, n_repeat=5):
    times = []
    rt_last = None
    for _ in range(n_repeat):
        t0 = perf_counter()
        rt_last = func(indices, n_src, order_dtype=order_dtype)
        _ = rt_last.row_splits.numpy()  # force materialization
        times.append(perf_counter() - t0)
    return np.array(times), rt_last

# Reuse n_src / indices from the performance cell above.
rng_access = np.random.default_rng(2027)

def benchmark_access(rt, n_repeat=30):
    times = []
    n_active_syn = None
    for _ in range(n_repeat):
        # fraction_of_active_neurons = np.random.uniform(0.0001, 0.05)
        fraction_of_active_neurons = np.abs(np.random.normal(loc=0.007, scale=0.05))
        active_pre_count = min(int(fraction_of_active_neurons * n_src), n_src)
        active_pre_ids = rng_access.choice(n_src, size=active_pre_count, replace=False)
        active_pre_ids_tf = tf.convert_to_tensor(active_pre_ids, dtype=tf.int32)
        t0 = perf_counter()
        active_rows = tf.gather(rt, active_pre_ids_tf)
        flat = active_rows.flat_values
        flat_np = flat.numpy()  # force materialization
        times.append(perf_counter() - t0)
        if n_active_syn is None:
            n_active_syn = int(flat_np.size)
    return np.array(times), n_active_syn




# # Build benchmark
# build_i32, rt_i32 = benchmark_build(indices, n_src, np.int32, make_pre_ind_table_with_dtype, n_repeat=1)
# build_i64, rt_i640 = benchmark_build(indices, n_src, np.int64, make_pre_ind_table_with_dtype0, n_repeat=1)
build_i32, rt_i321 = benchmark_build(indices, n_src, np.int32, make_pre_ind_table_with_dtype2, n_repeat=1)
# build_i32, rt_i320 = benchmark_build(indices, n_src, np.int32, make_pre_ind_table_with_dtype3, n_repeat=1)


# Access benchmark (active synapses)
access_i64, n_active_i64 = benchmark_access(rt_i640, n_repeat=300)
access_i320, n_active_i320 = benchmark_access(rt_i320, n_repeat=300)
access_i321, n_active_i321 = benchmark_access(rt_i321, n_repeat=300)

print('Ragged flat_values dtype (int64 build):', rt_i640.flat_values.dtype)
print('Ragged flat_values dtype (int32 build 1):', rt_i320.flat_values.dtype)
print('Ragged flat_values dtype (int32 build 2):', rt_i321.flat_values.dtype)
print('Active presynaptic neurons sampled:', active_pre_count)
print('Active synapses accessed (int64 table):', n_active_i64)
print('Active synapses accessed (int32 table 1):', n_active_i320)
print('Active synapses accessed (int32 table 2):', n_active_i321)
print()
print('Build int32:', build_i32, 'mean=', build_i32.mean())
print('Build int64:', build_i64, 'mean=', build_i64.mean())
print('Build speedup int32/int64:', build_i64.mean() / build_i32.mean())
print()
print('Access int64:', access_i64.mean())
print('Access int32 table 1:', access_i320.mean())
print('Access int32 table 2:', access_i321.mean())

# Access int64: 0.008858242189953671
# Access int32 table 1: 0.008609728223336787
# Access int32 table 2: 0.007859200699989136

Ragged flat_values dtype (int64 build): <dtype: 'int64'>
Ragged flat_values dtype (int32 build 1): <dtype: 'int64'>
Ragged flat_values dtype (int32 build 2): <dtype: 'int32'>
Active presynaptic neurons sampled: 800
Active synapses accessed (int64 table): 2368926
Active synapses accessed (int32 table 1): 2041493
Active synapses accessed (int32 table 2): 11657280

Build int32: [17.83428895] mean= 17.834288949999973
Build int64: [17.78906381] mean= 17.789063811999767
Build speedup int32/int64: 0.9974641468394396

Access int64: 0.01627785970329872
Access int32 table 1: 0.014471165989983395
Access int32 table 2: 0.01318680323331743


In [77]:
# Equivalence test: from_value_rowids vs from_row_splits

def build_group_indices_value_rowids(edge_type_ids):
    edge_type_ids = tf.convert_to_tensor(edge_type_ids, dtype=tf.int32)
    n_edges = tf.shape(edge_type_ids)[0]
    original_indices = tf.range(n_edges, dtype=tf.int32)
    sorted_indices = tf.argsort(edge_type_ids, axis=0)
    permuted_original_indices = tf.gather(original_indices, sorted_indices)
    sorted_edge_type_ids = tf.gather(edge_type_ids, sorted_indices)

    unique_types, row_indices = tf.unique(sorted_edge_type_ids)
    nrows = tf.shape(unique_types)[0]

    # Keep this on CPU so the check works even with GPU deterministic mode enabled.
    with tf.device('/CPU:0'):
        rt = tf.RaggedTensor.from_value_rowids(
            values=permuted_original_indices,
            value_rowids=row_indices,
            nrows=nrows,
        )
    return rt, sorted_edge_type_ids, permuted_original_indices


def build_group_indices_row_splits(edge_type_ids):
    edge_type_ids = tf.convert_to_tensor(edge_type_ids, dtype=tf.int32)
    n_edges = tf.shape(edge_type_ids)[0]
    original_indices = tf.range(n_edges, dtype=tf.int32)
    sorted_indices = tf.argsort(edge_type_ids, axis=0)
    permuted_original_indices = tf.gather(original_indices, sorted_indices)
    sorted_edge_type_ids = tf.gather(edge_type_ids, sorted_indices)

    _, _, counts = tf.unique_with_counts(sorted_edge_type_ids)
    row_splits = tf.concat(
        [tf.zeros((1,), dtype=counts.dtype), tf.cumsum(counts)],
        axis=0,
    )
    rt = tf.RaggedTensor.from_row_splits(
        values=permuted_original_indices,
        row_splits=row_splits,
        validate=False,
    )
    return rt, sorted_edge_type_ids, permuted_original_indices


# Edge cases
edge_type_cases = [
    np.array([], dtype=np.int32),
    np.array([0], dtype=np.int32),
    np.array([1, 1, 1, 1], dtype=np.int32),
    np.array([3, 0, 2, 0, 3, 1, 2, 1], dtype=np.int32),
]

for i, edge_type_ids in enumerate(edge_type_cases):
    rt_a, sorted_a, perm_a = build_group_indices_value_rowids(edge_type_ids)
    rt_b, sorted_b, perm_b = build_group_indices_row_splits(edge_type_ids)

    assert np.array_equal(sorted_a.numpy(), sorted_b.numpy()), f'Edge case {i}: sorted ids mismatch'
    assert np.array_equal(perm_a.numpy(), perm_b.numpy()), f'Edge case {i}: permuted indices mismatch'
    assert ragged_exact_equal(rt_a, rt_b), f'Edge case {i}: ragged mismatch'


# Randomized checks
rng = np.random.default_rng(4242)
random_trials = 200
for t in range(random_trials):
    n_edges = int(rng.integers(0, 20_000))
    n_types = int(rng.integers(1, 500))
    edge_type_ids = rng.integers(0, n_types, size=n_edges, dtype=np.int32)

    rt_a, sorted_a, perm_a = build_group_indices_value_rowids(edge_type_ids)
    rt_b, sorted_b, perm_b = build_group_indices_row_splits(edge_type_ids)

    if not np.array_equal(sorted_a.numpy(), sorted_b.numpy()):
        raise AssertionError(f'Random trial {t}: sorted ids mismatch')
    if not np.array_equal(perm_a.numpy(), perm_b.numpy()):
        raise AssertionError(f'Random trial {t}: permuted indices mismatch')
    if not ragged_exact_equal(rt_a, rt_b):
        raise AssertionError(f'Random trial {t}: ragged mismatch')

print(f'Passed {len(edge_type_cases)} edge cases and {random_trials} randomized trials.')
print('from_value_rowids and from_row_splits are exactly equivalent for this grouping path.')



Passed 4 edge cases and 200 randomized trials.
from_value_rowids and from_row_splits are exactly equivalent for this grouping path.
